In [ ]:
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
customer = pd.read_csv("D:\Project_Olist_clean\customer_cleaned.csv")
geolocation = pd.read_csv("D:\Project_Olist_clean\geolocation_cleaned.csv")
order_items= pd.read_csv("D:\Project_Olist_clean\order_items_cleaned.csv")
payments= pd.read_csv("D:\Project_Olist_clean\payments_cleaned.csv")
review=pd.read_csv("D:\\Project_Olist_clean\\review_cleaned.csv")
order=pd.read_csv("D:\Project_Olist_clean\order_cleaned.csv")
products=pd.read_csv("D:\Project_Olist_clean\products_cleaned.csv")
sellers=pd.read_csv("D:\Project_Olist_clean\sellers_cleaned.csv")
category=pd.read_csv("D:\Project_Olist_clean\category_cleaned.csv")

In [3]:
tables = {
    "customer": customer,
    "geolocation":geolocation,
    "order_items": order_items,
    "payments": payments,
    "review": review,
    "order":order,
    "products":products,
    "sellers": sellers,
    "category":category

}

### Tổng tiền mỗi order

In [4]:
order_value = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        order_value=("price", "sum"),
        item_count=("order_id", "size")
    )
)

order_value.head()

,order_id,order_value,item_count
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,1
1,00018f77f2f0320c557190d7a144bdd3,239.90,1
2,000229ec398224ef6ca0657da4fc703e,199.00,1
3,00024acbcdf0a6daa1e931b038114c75,12.99,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,1


### Tạo cột Delivery_date

In [15]:
order["order_purchase_timestamp"] = pd.to_datetime(
    order["order_purchase_timestamp"]
)

order["order_delivered_customer_date"] = pd.to_datetime(
    order["order_delivered_customer_date"]
)

order["delivery_days"] = (
    order["order_delivered_customer_date"]
    - order["order_purchase_timestamp"]
).dt.days

order.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_late,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0,8.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0,13.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0,9.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,0,13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,0,2.0


### Thêm cột is_late đánh giá đơn hàng có trễ hay ko?(0: ko trễ, 1: trễ)

In [17]:
order["order_estimated_delivery_date"] = pd.to_datetime(
    order["order_estimated_delivery_date"]
)
order["is_late"] = (
    order["order_delivered_customer_date"]
    > order["order_estimated_delivery_date"]
).astype("int")

order.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_late,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0,8.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0,13.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0,9.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,0,13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,0,2.0


### Merge order với customer ==> customer_oreder

In [18]:
customer_orders = order.merge(
    customer[
        [
            "customer_id",
            "customer_unique_id",
            "customer_city",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left"
)
customer_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_late,delivery_days,customer_unique_id,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0,8.0,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0,13.0,af07308b275d755c9edb36a90c618231,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0,9.0,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,0,13.0,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,0,2.0,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP


### Merge với order_value

In [20]:
customer_orders = customer_orders.merge(
    order_value,
    on="order_id",
    how="left"
)

### Thêm cột review cũng mỗi customer vào

In [ ]:
customer_orders = customer_orders.merge(
    review[
        [
            "order_id",
            "review_score"
        ]
    ],
    on="order_id",
    how="left"
)

customer_orders["review_score"] = customer_orders["review_score_y"]

customer_orders = customer_orders.drop(
    columns=["review_score_x", "review_score_y"]
)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_late,delivery_days,customer_unique_id,customer_city,customer_state,order_value_x,item_count_x,order_value_y,item_count_y,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0,8.0,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,29.99,1.0,29.99,1.0,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0,13.0,af07308b275d755c9edb36a90c618231,barreiras,BA,118.70,1.0,118.70,1.0,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0,9.0,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,159.90,1.0,159.90,1.0,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,0,13.0,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,45.00,1.0,45.00,1.0,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,0,2.0,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,19.90,1.0,19.90,1.0,5.0


In [ ]:
customer_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_late,delivery_days,customer_unique_id,customer_city,customer_state,order_value,item_count,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0,8.0,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,29.99,1.0,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0,13.0,af07308b275d755c9edb36a90c618231,barreiras,BA,118.70,1.0,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0,9.0,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,159.90,1.0,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,0,13.0,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,45.00,1.0,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,0,2.0,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,19.90,1.0,5.0


### Để mỗi customer_unique_id là 1 dòng chứa các chỉ số

In [26]:
customer_features = (
    customer_orders
    .groupby("customer_unique_id")
    .agg(
        order=("order_id", "nunique"),
        spend=("order_value", "sum"),
        avg_review=("review_score", "mean"),
        avg_delivery_days=("delivery_days", "mean"),
        late_orders=("is_late", "sum"),
        last_purchase=("order_purchase_timestamp", "max")
    )
    .reset_index()
)
customer_orders.head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_late,delivery_days,customer_unique_id,customer_city,customer_state,order_value,item_count,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0,8.0,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,29.99,1.0,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0,13.0,af07308b275d755c9edb36a90c618231,barreiras,BA,118.70,1.0,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0,9.0,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,159.90,1.0,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,0,13.0,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,45.00,1.0,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,0,2.0,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,19.90,1.0,5.0


In [27]:
customer_features["AOV"] = (
    customer_features["spend"]
    / customer_features["order"]
)

In [28]:
customer_features["repeat_customer"] = (
    customer_features["order"] > 1
).astype(int)

In [29]:
analysis_date = customer_orders["order_purchase_timestamp"].max()

customer_features["recency"] = (
    analysis_date
    - customer_features["last_purchase"]
).dt.days

In [30]:
customer_features["frequency"] = customer_features["order"]

In [31]:
customer_location = (
    customer[
        [
            "customer_unique_id",
            "customer_city",
            "customer_state"
        ]
    ]
    .drop_duplicates("customer_unique_id")
)

customer_features = customer_features.merge(
    customer_location,
    on="customer_unique_id",
    how="left"
)

In [33]:
customer_features.head()

,customer_unique_id,order,spend,avg_review,avg_delivery_days,late_orders,last_purchase,AOV,repeat_customer,recency,frequency,customer_city,customer_state
0,0000366f3b9a7992bf8c76cfdf3221e2,1,129.90,5.0,6.0,0,2018-05-10 10:56:27,129.90,0,160,1,cajamar,SP
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,18.90,4.0,3.0,0,2018-05-07 11:11:27,18.90,0,163,1,osasco,SP
2,0000f46a3911fa3c0805444483337064,1,69.00,3.0,25.0,0,2017-03-10 21:05:03,69.00,0,585,1,sao jose,SC
3,0000f6ccb0745a6a4b88665a16c9f078,1,25.99,4.0,20.0,0,2017-10-12 20:29:41,25.99,0,369,1,belem,PA
4,0004aac84e0df4da2b147fca70cf8255,1,180.00,5.0,13.0,0,2017-11-14 19:45:42,180.00,0,336,1,sorocaba,SP


In [34]:
customer_features["customer_unique_id"].duplicated().sum()

np.int64(0)

### Category và giá trị của đơn hàng ĐẦU TIÊN mỗi khách
Dùng để vẽ 2 chart: Repeat Rate theo category lần mua đầu, và Repeat Rate theo giá trị đơn đầu tiên.

In [ ]:
# 1) Chuẩn hóa tên category ở CẢ 2 bảng trước khi merge
#    (strip khoảng trắng dư + lowercase) để tránh merge bị lệch do khác định dạng
products["product_category_name"] = (
    products["product_category_name"]
    .astype("string")
    .str.strip()
    .str.lower()
)

category["product_category_name"] = (
    category["product_category_name"]
    .astype("string")
    .str.strip()
    .str.lower()
)

category["product_category_name_english"] = (
    category["product_category_name_english"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# 2) Map tay 2 category bị thiếu trong file translation gốc (lỗi đã biết của dataset Olist)
manual_category_map = {
    "pc_gamer": "pc_gamer",
    "portateis_cozinha_e_preparadores_de_alimentos": "kitchen_portables_and_food_preparers",
}

category_full = pd.concat(
    [
        category[["product_category_name", "product_category_name_english"]],
        pd.DataFrame(
            {
                "product_category_name": list(manual_category_map.keys()),
                "product_category_name_english": list(manual_category_map.values()),
            }
        ),
    ],
    ignore_index=True,
).drop_duplicates("product_category_name", keep="first")

# 3) Merge order_items -> products (category gốc) -> category_full (category tiếng Anh)
order_items_cat = order_items.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
)

order_items_cat = order_items_cat.merge(
    category_full,
    on="product_category_name",
    how="left"
)

# 4) Nếu vẫn còn thiếu (sản phẩm null category ngay từ products), gán "unknown"
#    thay vì để trống, và KHÔNG cho groupby tự loại bỏ nhóm NaN (dropna=False)
order_items_cat["product_category_name_english"] = (
    order_items_cat["product_category_name_english"].fillna("unknown")
)

order_category = (
    order_items_cat
    .groupby(["order_id", "product_category_name_english"], as_index=False, dropna=False)["price"]
    .sum()
    .sort_values("price", ascending=False)
    .drop_duplicates("order_id", keep="first")
    [["order_id", "product_category_name_english"]]
    .rename(columns={"product_category_name_english": "order_category"})
)

# 5) Kiểm tra lại: số dòng NaN category phải giảm mạnh, chỉ còn đúng phần products bị null gốc
print("NaN category sau khi sửa:", order_items_cat["product_category_name_english"].isna().sum())
print("Số order_id trong order_category:", order_category["order_id"].nunique())
order_category.head()

In [ ]:
# 2) Ghép category vào customer_orders (mỗi dòng order) để có category theo order_id
customer_orders = customer_orders.merge(
    order_category,
    on="order_id",
    how="left"
)

customer_orders.head()

In [ ]:
# 3) Lấy ĐƠN HÀNG ĐẦU TIÊN của mỗi khách (theo order_purchase_timestamp sớm nhất)
#    rồi giữ lại order_value và order_category của đơn đó
first_order = (
    customer_orders
    .sort_values("order_purchase_timestamp")
    .drop_duplicates("customer_unique_id", keep="first")
    [["customer_unique_id", "order_value", "order_category"]]
    .rename(columns={
        "order_value": "first_order_value",
        "order_category": "first_order_category"
    })
)

first_order.head()

In [ ]:
# 4) Merge 2 cột mới vào customer_features
customer_features = customer_features.merge(
    first_order,
    on="customer_unique_id",
    how="left"
)

customer_features[["customer_unique_id", "first_order_value", "first_order_category"]].head()

In [ ]:
# 5) Kiểm tra nhanh: không phát sinh thêm dòng, không có customer nào bị thiếu 2 cột mới
print("Số dòng customer_features:", len(customer_features))
print("Thiếu first_order_value:", customer_features["first_order_value"].isna().sum())
print("Thiếu first_order_category:", customer_features["first_order_category"].isna().sum())
customer_features["first_order_value"].describe()

In [35]:
customer_features.to_csv(
    r"D:\Project_Olis\cleaned\customer_unique_id.csv",
    index=False
)